<a href="https://colab.research.google.com/github/MarcelaAmbrozio/analise-ecommerce-roupas/blob/main/1_LimpezaDados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install pyspark

In [24]:
import pyspark
from pyspark.sql import SparkSession

In [25]:
spark = SparkSession.builder.getOrCreate()

In [26]:
produtos = spark.read.csv("produtos.csv", header=True)
vendedores = spark.read.csv("vendedores.csv", header=True)
clientes = spark.read.csv("clientes.csv", header=True)
itens_pedido = spark.read.csv("itens_pedido.csv", header=True)
pagamentos_pedido = spark.read.csv("pagamentos_pedido.csv", header=True)
avaliacoes_pedido = spark.read.csv("avaliacoes_pedido.csv", header=True)
pedidos = spark.read.csv("pedidos.csv", header=True)

In [27]:
print('dataframe produto: \n', produtos.show(n=5, truncate=False))
print('dataframe vendedores: \n', vendedores.show(n=5))
print('dataframe clientes: \n', clientes.show(n=5))
print('dataframe itens_pedido: \n', itens_pedido.show(n=5))
print('dataframe pagamentos_pedido: \n', pagamentos_pedido.show(n=5))
print('dataframe avaliacoes_pedido: \n', avaliacoes_pedido.show(n=5))
print('dataframe pedidos: \n', pedidos.show(n=5))

+--------------------------------+---------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|id_produto                      |categoria_produto    |tamanho_nome_produto|tamanho_descricao_produto|quantidade_fotos_produto|peso_produto_g|comprimento_produto_cm|altura_produto_cm|largura_produto_cm|
+--------------------------------+---------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                  |287                      |1                       |225           |16                    |10               |14                |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                  |276                      |1                       |1000          |30                    |18               |

In [28]:
# Acessar colunas
from pyspark.sql.functions import col

clientes.select('id_cliente').show(n=1, truncate=False)
clientes.select(col('id_cliente')).show(n=1)
clientes.select(clientes['id_cliente']).show(1)
clientes.select(clientes.id_cliente).show(1)

+--------------------------------+
|id_cliente                      |
+--------------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|
+--------------------------------+
only showing top 1 row
+--------------------+
|          id_cliente|
+--------------------+
|06b8999e2fba1a1fb...|
+--------------------+
only showing top 1 row
+--------------------+
|          id_cliente|
+--------------------+
|06b8999e2fba1a1fb...|
+--------------------+
only showing top 1 row
+--------------------+
|          id_cliente|
+--------------------+
|06b8999e2fba1a1fb...|
+--------------------+
only showing top 1 row


In [29]:
produtos_tratar_categ_nulos = produtos.na.fill({'categoria_produto': 'Não especificado'})
produtos_tratar_categ_nulos.filter(col('categoria_produto') == 'Não especificado').count()

610

In [30]:
print('Total de pedidos: ', pedidos.count())

pedidos_unicos = pedidos.dropDuplicates()
print('Total de pedidos únicos: ', pedidos_unicos.count())

pedidos_remocao_nulos = pedidos_unicos.na.drop()
print('Total de pedidos após limpeza de valaores nulos: ', pedidos_remocao_nulos.count())

pedidos_remocao_nulos_id = pedidos_unicos.na.drop(subset=['id_cliente', 'id_pedido'])
print('Total de pedidos após limpeza de id nulos', pedidos_remocao_nulos_id.count())

Total de pedidos:  99441
Total de pedidos únicos:  99441
Total de pedidos após limpeza de valaores nulos:  96461
Total de pedidos após limpeza de id nulos 99441


In [31]:
colunas = ['peso_produto_g', 'comprimento_produto_cm', 'altura_produto_cm', 'largura_produto_cm']

for coluna in colunas:
  produtos = produtos.na.fill({coluna: 0})

In [32]:
produtos.write.mode('overwrite').option('header', 'true').csv('produtos_tratados.csv')

In [33]:
spark.read.option('header', 'true').csv('produtos_tratados.csv').show(5)

+--------------------+--------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|          id_produto|   categoria_produto|tamanho_nome_produto|tamanho_descricao_produto|quantidade_fotos_produto|peso_produto_g|comprimento_produto_cm|altura_produto_cm|largura_produto_cm|
+--------------------+--------------------+--------------------+-------------------------+------------------------+--------------+----------------------+-----------------+------------------+
|1e9e8ef04dbcff454...|          perfumaria|                  40|                      287|                       1|           225|                    16|               10|                14|
|3aa071139cb16b67c...|               artes|                  44|                      276|                       1|          1000|                    30|               18|                20|
|96bd76ec8810374ed...|       esporte_lazer|  